# RetailMind AI — Day 16: Product Image Variations and Promo Video

Use a catalog image as the initialization image for controlled Stable Diffusion img2img variations. Sweep `strength` and `guidance_scale`, review the saved benchmark, then assemble a short crossfade MP4.

Install `requirements.txt` first and ensure the chosen product photo exists. Model weights download on first use; CUDA is preferred. Generation is opt-in in this notebook. Pixel MAE/PSNR are simple visual similarity diagnostics, not semantic product-accuracy scores.

In [ ]:
from pathlib import Path
from IPython.display import display, Video
from PIL import Image
from src.day16_image_variations_video import (
    create_variations, assemble_promo_video, DEFAULT_OUTPUT_DIR,
)

ROOT = Path.cwd()
IMAGE_PATH = next((ROOT / 'data' / 'fashion-product-images-small' / 'images').glob('*.jpg'))
PRODUCT = {
    'product_name': 'Everyday Tote',
    'category': 'canvas tote bag',
    'style': 'minimal, cheerful, warm colors',
    'setting': 'sunlit city market',
    'brand_voice': 'useful, optimistic, approachable',
}
STRENGTHS = [0.25, 0.45, 0.65]
GUIDANCE_SCALES = [6.5, 7.5]
SEED = 42
display(Image.open(IMAGE_PATH).convert('RGB'))
print(f'Catalog image: {IMAGE_PATH}')

## 1. Compare strength and guidance settings

Lower strength usually keeps the generated image closer to the starting image; higher strength gives the model more freedom. The sweep writes a row for each setting combination, including elapsed generation time and pixel similarity diagnostics.

In [ ]:
GENERATE_ASSETS = False
if GENERATE_ASSETS:
    asset_images, benchmark_rows, benchmark_path, metadata_path = create_variations(
        IMAGE_PATH, **PRODUCT, strengths=STRENGTHS,
        guidance_scales=GUIDANCE_SCALES, seed=SEED,
    )
    print(f'Benchmark: {benchmark_path}')
    print(f'Metadata: {metadata_path}')
    for image in asset_images[1:]:
        display(image)
else:
    print('Image generation skipped. Set GENERATE_ASSETS = True to load the model and run the sweep.')

## 2. Inspect the variation benchmark

Generation time is hardware-dependent. Pixel metrics are useful for comparing how strongly an output differs from its source, but human review is required to judge whether the product identity and campaign quality are preserved.

In [ ]:
if GENERATE_ASSETS:
    import pandas as pd
    benchmark = pd.read_csv(benchmark_path)
    display(benchmark.sort_values(['strength', 'guidance_scale']))
else:
    print('Run the sweep above to populate the benchmark table.')

## 3. Assemble the promo clip

The clip crossfades the input and all generated variations. Curate the best outputs first if you want a campaign-ready sequence.

In [ ]:
if GENERATE_ASSETS:
    video_path = assemble_promo_video(asset_images, DEFAULT_OUTPUT_DIR / 'promo_clip.mp4')
    print(f'Saved promo clip: {video_path}')
    display(Video(str(video_path), embed=True))
else:
    print('Video assembly skipped until variations have been generated.')